# How the $W_{qk}$ matrices evolve — full batch, zero init

Analysis of the runs produced by `train_zero_init_25K_points.py`: **Adam, batch = 25,000** (the whole
training set), so **1 epoch = 1 optimizer step** and every recorded gradient is the **exact
population gradient**, not a 256-sample estimate.

Recorded per threshold: $W_{qk}$ for both layers at every step, and $\partial L/\partial W_{qk}$ for
the first `RECORD_GRAD_STEPS`. Row convention: `W[k]` is the state **before** step $k{+}1$ (so `W[0]`
is the zero init), and `G[k]` is the gradient evaluated **at** `W[k]`.

Several learning rates have been run. **Set `RUN_TAG` in the setup cell** to choose one; leaving it
`None` picks whatever the training script is currently configured for. Section 1 lists everything
found on disk, and every number in this notebook is computed from the run you select — nothing is
hardcoded in the prose.

## What to look for

Three things, and the second is the one that decides whether the run supports the "GD builds the
construction" claim:

1. **The gating result.** $\partial L/\partial W^1_{qk}$ should be *identically* zero at step 1 —
   every entry, every $\alpha$ — because $W^2$ enters $\partial S^2/\partial Z^1$ as a multiplicative
   factor. This is a statement about the loss, so the exact gradient cannot soften it (§2).

2. **Loss versus structure.** CE falling is not the same as the induction circuit forming.
   `induction_diag_off` near **1.0** means the signal $4\times4$ diagonal is indistinguishable from
   its background — no circuit — however good the loss looks. Watch it against the cosine to the
   hand construction (§4, §8).

3. **When layer 1 overtakes layer 2.** At small learning rates layer 1 trails for hundreds of steps.
   At large ones it passes layer 2 within a handful, because Adam steps ~lr per coordinate once the
   gate is open, regardless of gradient size (§3).

The animations show each matrix **twice** — raw, and normalised to unit Frobenius norm — because the
scale can grow by three orders of magnitude while the pattern is what actually matters. One fixed
colour scale would leave the early frames blank; a per-frame scale would hide the growth entirely.

**Contents**

1. Load the run (and the headline numbers)
2. Step 1 — the gating result under the exact gradient
3. Scale: how the norms grow, and the layer-1 overtake
4. Pattern: settling or thrashing?
5. **Animation 1** — scale and pattern side by side
6. **Animation 2** — weights and the exact gradient
7. **Animation 3** — the induction block, cell by cell
8. Did it build the circuit?
9. Takeaways

## 0. Setup

In [ ]:
import os, json, math, importlib.util
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.colors import SymLogNorm
from IPython.display import HTML, display

V, VS, T = 8, 4, 100
D, IN2   = V + T, 2 * (V + T)
SD       = D ** 0.5
ALPHA_REC, GAMMA, P1, P2, NF, BIG = 20.0, -50000.0, 1000.0, 50.0, 10.0, -1e9

THR  = 0.6          # threshold shown in the single-alpha panels and animations
SEED = 0

RUN_TAG = None      # e.g. 'wqkzero25k_opt-adam_lr-3.0_b-25000_init-zeros_ep-100'
                    # None -> whatever train_zero_init_25K_points.py is currently set to

_s = importlib.util.spec_from_file_location('Z25', 'train_zero_init_25K_points.py')
Z25 = importlib.util.module_from_spec(_s); _s.loader.exec_module(Z25)
THRS = Z25.THRESHOLDS
TAG  = RUN_TAG or Z25.cell_tag()
RDIR = os.path.join(Z25.CFG['runs_dir'], TAG)
ADIR = os.path.join(Z25.CFG['art_dir'],  TAG)

plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 140, 'font.size': 11, 'axes.grid': False,
                     'animation.embed_limit': 300})
C1, C2 = '#d62728', '#1f77b4'
FIGDIR = 'artifacts/figures_full_25k'
os.makedirs(FIGDIR, exist_ok=True)
SUF = ''            # set after TAG is known, below

# ---- the construction, as the reference target ----
Wq1 = torch.eye(D) * SD
T1  = torch.zeros(D, D)
for k in range(VS, V):
    T1[k, :] = BIG
for tq in range(T):
    for tk in range(T):
        T1[V + tk, V + tq] = BIG if tq == tk else -ALPHA_REC * (tq - tk)
Wq2 = torch.zeros(D, IN2)
Wq2[:VS, :VS]      = torch.eye(VS) * SD * P1
Wq2[:VS, D:D + VS] = torch.eye(VS) * SD * P2
for v in range(V):
    Wq2[VS, v] = SD * NF
Wk2 = torch.zeros(D, IN2)
Wk2[:V, D:D + V] = torch.eye(V)
for v in range(VS, V):
    Wk2[VS, v] = GAMMA
CQK  = [(Wq1.t() @ T1.t()).numpy(), (Wq2.t() @ Wk2).numpy()]
MASK = [np.zeros((D, D), bool), np.zeros((IN2, IN2), bool)]
MASK[0][:, VS:V] = True; MASK[0][V:D, V:D] = True
MASK[1][0:V, 0:V] = True; MASK[1][0:V, D:D + V] = True; MASK[1][D:D + V, D:D + V] = True

def bounds(n):
    return [V] if n == D else ([V, D, D + V] if n == IN2 else [])

def heat(ax, A, title, vmax=None, lw=0.7, fs=10):
    A = np.asarray(A, dtype=np.float64)
    v = float(vmax) if vmax else float(np.abs(A).max())
    if v <= 0:
        ax.imshow(A, cmap='RdBu_r', interpolation='nearest', vmin=-1, vmax=1)
    else:
        ax.imshow(A, cmap='RdBu_r', interpolation='nearest',
                  norm=SymLogNorm(linthresh=v / 1e3, linscale=0.6, vmin=-v, vmax=v, base=10))
    for b in bounds(A.shape[0]): ax.axhline(b - .5, color='k', lw=lw)
    for b in bounds(A.shape[1]): ax.axvline(b - .5, color='k', lw=lw)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=fs)

def annot_block(ax, A, title, vmax=None, fs=7.5):
    A = np.asarray(A, dtype=np.float64)
    v = float(vmax) if vmax else float(np.abs(A).max())
    if v <= 0:
        ax.imshow(A, cmap='RdBu_r', interpolation='nearest', vmin=-1, vmax=1)
    else:
        ax.imshow(A, cmap='RdBu_r', interpolation='nearest',
                  norm=SymLogNorm(linthresh=v / 1e3, linscale=0.6, vmin=-v, vmax=v, base=10))
    for i in range(A.shape[0]):
        for j in range(A.shape[1]):
            ax.text(j, i, '.' if A[i, j] == 0 else f'{A[i, j]:.3g}',
                    ha='center', va='center', fontsize=fs)
    ax.set_xticks(range(A.shape[1])); ax.set_yticks(range(A.shape[0]))
    ax.set_xticklabels([f'k{i}' for i in range(A.shape[1])], fontsize=8)
    ax.set_yticklabels([f'q{i}' for i in range(A.shape[0])], fontsize=8)
    ax.set_title(title, fontsize=11)

def cos(a, b):
    a = np.asarray(a, dtype=np.float64).ravel(); b = np.asarray(b, dtype=np.float64).ravel()
    n = np.linalg.norm(a) * np.linalg.norm(b)
    return float(a @ b / n) if n > 0 else float('nan')

def share(A, mask):
    A = np.asarray(A, dtype=np.float64); tot = float((A ** 2).sum())
    return float((A[mask] ** 2).sum() / tot) if tot > 0 else float('nan')

def unit(A):
    '''W / ||W||_F -- the PATTERN, with the exploding scale divided out.'''
    A = np.asarray(A, dtype=np.float64); n = np.linalg.norm(A)
    return A / n if n > 0 else A

def diag_off(blk):
    a = np.abs(np.asarray(blk, dtype=np.float64))
    dg = np.diag(a).mean()
    off = (a.sum() - np.diag(a).sum()) / (a.size - a.shape[0])
    return dg / off if off > 1e-30 else np.nan

import glob as _glob
_found = sorted(os.path.basename(p.rstrip('/'))
                for p in _glob.glob(os.path.join(Z25.CFG['runs_dir'], 'wqkzero25k_*'))
                if os.path.isdir(p))
print('runs found on disk:')
for f in _found:
    print(f'  {"->" if f == TAG else "  "} {f}')
SUF = TAG.replace('wqkzero25k_opt-adam_', '').replace('_b-25000_init-zeros', '')
print(f'\nanalysing : {TAG}')
if RUN_TAG is None:
    print('(taken from the training script; set RUN_TAG above to pin a different one)')
if TAG not in _found:
    raise FileNotFoundError(f'{TAG} is not on disk -- set RUN_TAG to one of the above')

## 1. Load the run

In [ ]:
MET, SM, W, G = {}, {}, {}, {}
missing = []
for t in THRS:
    mp = os.path.join(RDIR, f'metrics_{Z25.ARCH}_seed{SEED}_x_{t}.json')
    sp = os.path.join(ADIR, f'snapmeta_x_{t}.json')
    if not (os.path.exists(mp) and os.path.exists(sp)):
        missing.append(mp); continue
    MET[t], SM[t] = json.load(open(mp)), json.load(open(sp))
    nw, ng = SM[t]['w_rows_written'], SM[t]['g_rows_written']
    W[t] = [np.load(os.path.join(ADIR, f'wqk{i}_x_{t}.npy'))[:nw].astype(np.float64) for i in (1, 2)]
    G[t] = [np.load(os.path.join(ADIR, f'gqk{i}_x_{t}.npy'))[:ng].astype(np.float64) for i in (1, 2)]
if missing:
    raise FileNotFoundError('run `python train_zero_init_25K_points.py` first; missing:\n  '
                            + '\n  '.join(missing))

NW, NG = len(W[THR][0]), len(G[THR][0])
sw, sg = np.arange(NW), np.arange(1, NG + 1)
wn = {t: [np.linalg.norm(W[t][i].reshape(len(W[t][i]), -1), axis=1) for i in (0, 1)] for t in THRS}
gn = {t: [np.linalg.norm(G[t][i].reshape(len(G[t][i]), -1), axis=1) for i in (0, 1)] for t in THRS}

print(f'{"alpha":>6} {"steps":>6} {"W rows":>7} {"G rows":>7} {"train CE":>18} {"test CE":>18} '
      f'{"min":>5}')
for t in THRS:
    m, h = MET[t], MET[t]['history']
    print(f'{t:>6} {m["steps_run"]:>6} {SM[t]["w_rows_written"]:>7} {SM[t]["g_rows_written"]:>7} '
          f'{h["train_loss"][0]:>8.5f} -> {m["final_train"]:<7.5f} '
          f'{h["test_loss"][0]:>8.5f} -> {m["final_test"]:<7.5f} {m["time_s"]/60:>5.1f}')

print('\nzero-init sanity: W row 0 exactly 0, and layer 1 unchanged over step 1')
for t in THRS:
    print(f'  alpha={t}: |W1(0)|max={np.abs(W[t][0][0]).max():.3g}  '
          f'|W2(0)|max={np.abs(W[t][1][0]).max():.3g}  '
          f'W1[0]==W1[1]: {np.array_equal(W[t][0][0], W[t][0][1])}')

### Headline numbers for this run

Computed, not asserted — these are the figures the rest of the notebook unpacks.

In [ ]:
b1 = MASK[1].sum() / MASK[1].size
print(f'{"alpha":>6} {"train CE":>19} {"|g1| step1":>12} {"ind_diag_off":>13} '
      f'{"cos->constr L2":>15} {"|W1|/|W2|":>10}')
for t in THRS:
    m, h = MET[t], MET[t]['history']
    blk = W[t][1][-1][:VS, D:D + VS]
    a = np.abs(blk); dg = np.diag(a).mean()
    off = (a.sum() - np.diag(a).sum()) / (a.size - VS)
    print(f'{t:>6} {h["train_loss"][0]:>8.4f} -> {m["final_train"]:<8.4f} '
          f'{np.linalg.norm(G[t][0][0]):>12.2e} {dg/off:>13.3f} '
          f'{cos(W[t][1][-1], CQK[1]):>15.4f} '
          f'{wn[t][0][-1]/wn[t][1][-1]:>10.2f}')
print('\nind_diag_off ~ 1.0 means NO diagonal structure in the induction block, whatever')
print('the loss did. Compare against cos->constr in the same row.')

## 2. Step 1 — the gating result under the exact gradient

$\partial L/\partial W^1_{qk}=0$ is an identity about the loss (the factor $W^2$ sits
multiplicatively inside $\partial S^2/\partial Z^1$), so it cannot be an artefact of batch noise.
With the full 25,000 sequences there is nowhere for a small residual to hide.

In [ ]:
print(f'{"alpha":>6} {"||g1|| step1":>14} {"nonzero g1":>11} {"||g2|| step1":>14} '
      f'{"nonzero g2":>11} {"||g1|| step2":>14}')
for t in THRS:
    g1, g2 = G[t][0][0], G[t][1][0]
    print(f'{t:>6} {np.linalg.norm(g1):>14.4e} {int((g1 != 0).sum()):>11,} '
          f'{np.linalg.norm(g2):>14.4e} {int((g2 != 0).sum()):>11,} '
          f'{np.linalg.norm(G[t][0][1]):>14.4e}')
print('\nlayer 1 is identically zero at step 1 and nonzero at step 2: the gate opens once W2 != 0.')

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
heat(axes[0], G[THR][0][0], r'$\partial L/\partial W^1_{qk}$ at the zero init (exact, 25k)'
                            '\nidentically zero — the flat panel IS the result')
heat(axes[1], G[THR][1][0], r'$\partial L/\partial W^2_{qk}$ at the zero init (exact, 25k)')
fig.suptitle(f'Step-1 population gradients   ($\\alpha$={THR})', fontsize=14)
fig.tight_layout(); fig.savefig(f'{FIGDIR}/step1_grad_{SUF}_a{THR}.png', bbox_inches='tight')
plt.show()

## 3. Scale — and the layer-1 overtake

At lr = 0.005 layer 1 trails layer 2 for hundreds of steps. At large learning rates it overtakes
within a handful and ends several times larger. Adam is the reason: it steps ~lr per coordinate
regardless of gradient magnitude, so once layer 1's gate is open its 11,664 coordinates advance at
the same per-coordinate rate as layer 2's 46,656 — and layer 1's are more consistently signed. The
table below reports the crossing step for this run.

In [ ]:
print(f'{"alpha":>6} {"overtake step":>14} {"|W1| final":>12} {"|W2| final":>12} {"ratio":>8}')
for t in THRS:
    a, b = wn[t][0], wn[t][1]
    ov = int(np.argmax(a > b)) if (a > b).any() else -1
    print(f'{t:>6} {ov:>14} {a[-1]:>12.1f} {b[-1]:>12.1f} {a[-1]/b[-1]:>8.2f}')

fig, axes = plt.subplots(1, 3, figsize=(19, 5.0))
ls = {0.6: '-', 0.8: '--', 1.0: ':'}
for t in THRS:
    axes[0].plot(sw, wn[t][0], color=C1, ls=ls[t], lw=1.5, label=f'layer 1  a={t}')
    axes[0].plot(sw, wn[t][1], color=C2, ls=ls[t], lw=1.5, label=f'layer 2  a={t}')
    axes[1].plot(sw, wn[t][0] / np.maximum(wn[t][1], 1e-30), ls=ls[t], lw=1.6, label=f'a={t}')
    p = gn[t][0] > 0
    axes[2].semilogy(sg[p], gn[t][0][p], color=C1, ls=ls[t], lw=1.4, label=f'layer 1  a={t}')
    axes[2].semilogy(sg, gn[t][1], color=C2, ls=ls[t], lw=1.4, label=f'layer 2  a={t}')
axes[0].set_title(r'$\|W_{qk}\|_F$'); axes[0].set_xlabel('optimizer step')
axes[1].axhline(1.0, color='k', ls='--', lw=0.9)
axes[1].set_title(r'$\|W_1\|/\|W_2\|$   (crosses 1 within ~5-7 steps)')
axes[1].set_xlabel('optimizer step')
axes[2].set_title(r'exact $\|\partial L/\partial W_{qk}\|_F$   (step 1 layer 1 = 0, off scale)')
axes[2].set_xlabel('optimizer step')
for a in axes:
    a.grid(alpha=0.3); a.legend(fontsize=8)
fig.suptitle(f'Scale over {NW-1} full-batch steps   (Adam lr={Z25.CFG["lr"]}, '
             f'batch={Z25.CFG["batch_size"]:,})', fontsize=14)
fig.tight_layout(); fig.savefig(f'{FIGDIR}/scale_and_overtake_{SUF}.png', bbox_inches='tight')
plt.show()

## 4. Pattern — settling or thrashing?

Norms only tell you how far the weights travelled, not whether they are converging on a *shape*.
Two scale-free diagnostics:

* **step-to-step cosine** $\cos\!\big(W(k),W(k{+}1)\big)$ — how much the direction changes per step.
  Rising towards 1 means the pattern is locking in; staying low means each step is redirecting the
  solution, i.e. the learning rate is outrunning the gradient field.
* **cosine to the construction**, and the share of squared mass inside the construction's support
  (192 of 46,656 cells for layer 2 — a 0.41% baseline).

In [ ]:
step_cos = {t: [np.array([cos(W[t][i][k], W[t][i][k + 1]) for k in range(1, NW - 1)])
                for i in (0, 1)] for t in THRS}
constr_cos = {t: [np.array([cos(W[t][i][k], CQK[i]) for k in range(NW)]) for i in (0, 1)]
              for t in THRS}
supp = {t: [np.array([share(W[t][i][k], MASK[i]) for k in range(NW)]) for i in (0, 1)]
        for t in THRS}

fig, axes = plt.subplots(1, 3, figsize=(19, 5.0))
for t in THRS:
    axes[0].plot(np.arange(1, NW - 1), step_cos[t][1], ls=ls[t], lw=1.5, label=f'layer 2  a={t}')
    axes[1].plot(sw, constr_cos[t][1], ls=ls[t], lw=1.5, label=f'layer 2  a={t}')
    axes[2].plot(sw, supp[t][1], ls=ls[t], lw=1.5, label=f'layer 2  a={t}')
axes[0].axhline(1.0, color='k', ls='--', lw=0.9)
axes[0].set_title(r'step-to-step $\cos(W_2(k), W_2(k{+}1))$' '\n1 = direction locked in')
axes[1].axhline(0.0, color='k', ls='--', lw=0.9)
axes[1].set_title(r'$\cos(W_2(k),\ W_2^{\rm construction})$')
axes[2].axhline(MASK[1].sum() / MASK[1].size, color='k', ls='--', lw=1.0,
                label=f'uniform baseline ({MASK[1].sum()/MASK[1].size:.2%})')
axes[2].set_title("share of $W_2$ mass in the construction's support")
for a in axes:
    a.set_xlabel('optimizer step'); a.grid(alpha=0.3); a.legend(fontsize=8)
fig.tight_layout(); fig.savefig(f'{FIGDIR}/pattern_diagnostics_{SUF}.png', bbox_inches='tight')
plt.show()

print(f'{"alpha":>6} {"step-cos @end":>14} {"cos to constr":>14} {"support share":>14} '
      f'{"x baseline":>11}')
for t in THRS:
    print(f'{t:>6} {step_cos[t][1][-1]:>14.4f} {constr_cos[t][1][-1]:>14.4f} '
          f'{supp[t][1][-1]:>14.4f} {supp[t][1][-1]/b1:>10.1f}x')

## 5. Animation 1 — scale and pattern side by side

The scale grows from 0 to thousands, so a single fixed colour scale would render the first frames
blank and a per-frame scale would hide the growth entirely. Every animation here therefore shows
each matrix **twice**:

* **top row, raw $W_{qk}$** on one fixed symlog scale across all frames — this is the *scale* story;
* **bottom row, $W_{qk}/\|W_{qk}\|_F$** on its own fixed scale — this is the *pattern* story, with
  the exploding magnitude divided out. If the circuit is forming, it forms here.

The right-hand column tracks where you are in the run. Frames are dense over the first 20 steps
(where everything happens) and sparser after.

In [ ]:
FRAMES = list(range(0, 21)) + list(range(22, NW, 4))      # dense early, sparse late
print(f'{len(FRAMES)} frames out of {NW} steps')

def scale_pattern_anim(t, fname, interval=380):
    Wl = W[t]
    raw_v  = [float(np.abs(Wl[i][FRAMES]).max()) for i in (0, 1)]
    uni    = [np.stack([unit(Wl[i][k]) for k in FRAMES]) for i in (0, 1)]
    uni_v  = [float(np.abs(uni[i]).max()) for i in (0, 1)]
    h = MET[t]['history']
    fig, axes = plt.subplots(2, 3, figsize=(16.5, 10.6),
                             gridspec_kw={'width_ratios': [1, 1, 1.15]})
    fig.subplots_adjust(top=0.90)

    def draw(fi):
        k = FRAMES[fi]
        for ax in axes.ravel():
            ax.clear()
        heat(axes[0][0], Wl[0][k], r'$W^1_{qk}$  raw' f'\n$\\|\\cdot\\|_F$ = {wn[t][0][k]:.1f}',
             vmax=raw_v[0])
        heat(axes[0][1], Wl[1][k], r'$W^2_{qk}$  raw' f'\n$\\|\\cdot\\|_F$ = {wn[t][1][k]:.1f}',
             vmax=raw_v[1])
        heat(axes[1][0], uni[0][fi], r'$W^1_{qk}/\|W^1_{qk}\|_F$   (pattern)', vmax=uni_v[0])
        heat(axes[1][1], uni[1][fi], r'$W^2_{qk}/\|W^2_{qk}\|_F$   (pattern)', vmax=uni_v[1])

        ax = axes[0][2]
        ax.plot(sw, wn[t][0], color=C1, lw=1.4, label='layer 1')
        ax.plot(sw, wn[t][1], color=C2, lw=1.4, label='layer 2')
        ax.plot([k], [wn[t][0][k]], 'o', color=C1, ms=9)
        ax.plot([k], [wn[t][1][k]], 'o', color=C2, ms=9)
        ax.axvline(k, color='k', lw=0.8, alpha=0.35)
        ax.set_title(r'$\|W_{qk}\|_F$', fontsize=10); ax.legend(fontsize=8); ax.grid(alpha=0.3)

        ax = axes[1][2]
        ax.plot(h['step'], h['train_loss'], color='#2ca02c', lw=1.4, label='train CE')
        ax.plot(h['step'], h['test_loss'], color='#9467bd', lw=1.2, ls='--', label='test CE')
        ax.plot([k], [h['train_loss'][k]], 'o', color='#2ca02c', ms=9)
        ax.axvline(k, color='k', lw=0.8, alpha=0.35)
        ax.set_title(f'cross-entropy    step {k}: {h["train_loss"][k]:.4f}', fontsize=10)
        ax.set_xlabel('optimizer step'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

        note = '  --  layer 1 frozen (gradient identically zero)' if k <= 1 else ''
        fig.suptitle(f'$\\alpha$={t}   step {k} / {NW-1}   '
                     f'(top: raw, fixed scale — bottom: unit-norm, the pattern){note}', fontsize=13)
        return []

    anim = animation.FuncAnimation(fig, draw, frames=len(FRAMES), interval=interval, blit=False)
    anim.save(fname, writer=animation.PillowWriter(fps=max(1, int(1000 / interval))))
    plt.close(fig)
    return anim

anim1 = scale_pattern_anim(THR, f'{FIGDIR}/evolution_scale_pattern_{SUF}_a{THR}.gif')
print(f'saved {FIGDIR}/evolution_scale_pattern_a{THR}.gif')
display(HTML(anim1.to_jshtml()))

## 6. Animation 2 — weights and the exact gradient

The first 50 steps, where gradients were recorded. Left column the weights, right column the exact
population gradient that produced the next update. Frame 0 is the point of the whole study: the
layer-1 gradient panel is blank while layer 2 already carries structure.

In [ ]:
def weight_grad_anim(t, fname, interval=400):
    n = NG
    vW = [float(np.abs(W[t][i][:n]).max()) for i in (0, 1)]
    vG = [float(np.abs(G[t][i][:n]).max()) for i in (0, 1)]
    fig, axes = plt.subplots(2, 2, figsize=(11.5, 11.0))
    fig.subplots_adjust(top=0.90)

    def draw(k):
        for ax in axes.ravel():
            ax.clear()
        heat(axes[0][0], W[t][0][k], r'$W^1_{qk}$', vmax=vW[0])
        heat(axes[0][1], W[t][1][k], r'$W^2_{qk}$', vmax=vW[1])
        heat(axes[1][0], G[t][0][k], r'$\partial L/\partial W^1_{qk}$  (exact)', vmax=vG[0])
        heat(axes[1][1], G[t][1][k], r'$\partial L/\partial W^2_{qk}$  (exact)', vmax=vG[1])
        note = '  --  layer 1 frozen: gradient identically zero' if k == 0 else ''
        fig.suptitle(f'$\\alpha$={t}   step {k}  '
                     f'(weights before update {k+1}, exact gradient at those weights){note}',
                     fontsize=13)
        return []

    anim = animation.FuncAnimation(fig, draw, frames=n, interval=interval, blit=False)
    anim.save(fname, writer=animation.PillowWriter(fps=max(1, int(1000 / interval))))
    plt.close(fig)
    return anim

anim2 = weight_grad_anim(THR, f'{FIGDIR}/evolution_weight_grad_{SUF}_a{THR}.gif')
print(f'saved {FIGDIR}/evolution_weight_grad_a{THR}.gif')
display(HTML(anim2.to_jshtml()))

## 7. Animation 3 — the induction block, cell by cell

$W^2_{qk}[0{:}8,\,108{:}116]$ — own token $\times$ key predecessor. The construction fills the
signal $4\times4$ corner with $10392.3\,I_4$ plus a $103.9$ trigger in column 4, and zero elsewhere.
What matters is the *relative* pattern: does the diagonal separate from its background?

Left the weights, right the exact gradient driving them.

In [ ]:
Wb = W[THR][1][:NG, :V, D:D + V]
Gb = G[THR][1][:NG, :V, D:D + V]
vW, vG = float(np.abs(Wb).max()), float(np.abs(Gb).max())
fig, axes = plt.subplots(1, 2, figsize=(13.5, 6.4))
fig.subplots_adjust(top=0.86)

def draw3(k):
    for ax in axes:
        ax.clear()
    annot_block(axes[0], Wb[k], r'$W^2_{qk}[0{:}8,\ 108{:}116]$', vmax=vW)
    annot_block(axes[1], Gb[k], r'$\partial L/\partial W^2_{qk}$  (exact, same block)', vmax=vG)
    fig.suptitle(f'induction block, $\\alpha$={THR}, step {k}   '
                 f'(diag/off-diag = {diag_off(Wb[k][:VS, :VS]):.3f};  '
                 f'construction target: $10392.3\\,I_4$)', fontsize=12)
    return []

anim3 = animation.FuncAnimation(fig, draw3, frames=NG, interval=420, blit=False)
anim3.save(f'{FIGDIR}/evolution_induction_{SUF}_a{THR}.gif', writer=animation.PillowWriter(fps=2))
plt.close(fig)
print(f'saved {FIGDIR}/evolution_induction_a{THR}.gif')
display(HTML(anim3.to_jshtml()))

## 8. Did it build the circuit?

Loss and structure can disagree, and this section reports both without splitting the difference.

`induction_diag_off` is mean$|$diagonal$|$ over mean$|$off-diagonal$|$ of the signal $4\times4$
sub-block (`qk_stats`, the same metric the training script logs). **1.0 means no diagonal
structure at all**; the construction is a pure diagonal there.

In [ ]:
print(f'{"alpha":>6} {"train CE":>18} {"ind_diag_off":>13} {"cos to constr L2":>17} '
      f'{"support x base":>15}')
for t in THRS:
    m, h = MET[t], MET[t]['history']
    blk = W[t][1][-1][:VS, D:D + VS]
    print(f'{t:>6} {h["train_loss"][0]:>8.4f} -> {m["final_train"]:<7.4f} '
          f'{diag_off(blk):>13.3f} {constr_cos[t][1][-1]:>17.4f} '
          f'{supp[t][1][-1]/b1:>14.1f}x')

print('\nthe signal 4x4 of the final induction block (diagonal vs its background):')
for t in THRS:
    blk = W[t][1][-1][:VS, D:D + VS]
    off = np.abs(blk - np.diag(np.diag(blk))).sum() / (blk.size - VS)
    print(f'  alpha={t}: diag {np.round(np.diag(blk), 1)}   mean |off-diag| {off:.1f}')

fig, axes = plt.subplots(2, 1 + len(THRS), figsize=(5.6 * (1 + len(THRS)), 11), squeeze=False)
for li in (0, 1):
    heat(axes[li][0], CQK[li], f'construction   layer {li+1}')
    for j, t in enumerate(THRS):
        heat(axes[li][j + 1], W[t][li][-1],
             f'learnt   layer {li+1}   $\\alpha$={t}\ncos = {constr_cos[t][li][-1]:+.4f}   '
             f'$\\|\\cdot\\|_F$ = {wn[t][li][-1]:.0f}')
fig.suptitle(f'Final solution after {NW-1} full-batch steps vs the hand construction '
             f'(per-panel symlog)', fontsize=14)
fig.tight_layout(); fig.savefig(f'{FIGDIR}/final_vs_construction_{SUF}.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 1 + len(THRS), figsize=(6.4 * (1 + len(THRS)), 6.2), squeeze=False)
annot_block(axes[0][0], CQK[1][:V, D:D + V], 'construction', fs=6.5)
for j, t in enumerate(THRS):
    blk = W[t][1][-1][:V, D:D + V]
    annot_block(axes[0][j + 1], blk,
                f'learnt, $\\alpha$={t}\ndiag/off-diag = {diag_off(blk[:VS, :VS]):.3f}', fs=6.5)
fig.suptitle(r'Final induction block $W^2_{qk}[0{:}8,\ 108{:}116]$', fontsize=13)
fig.tight_layout(); fig.savefig(f'{FIGDIR}/final_induction_block_{SUF}.png', bbox_inches='tight')
plt.show()

## 9. Takeaways

Read these against the tables above — the numbers are for whichever `RUN_TAG` you selected.

1. **The gating result is confirmed at population scale.** $\partial L/\partial W^1_{qk}=0$ at
   step 1, all 11,664 entries, at every $\alpha$, with the exact 25,000-sequence gradient. Layer 1
   is *gated* by $W^2_{qk}$, not merely slow. Batch-256 estimates were not hiding a residual.

2. **Layer 1 overtakes layer 2 early at these learning rates** — the reverse of the lr = 0.005
   ordering, where it was still at 0.42x after 20 steps. The gate opens once and then stops being
   the bottleneck; what remains is Adam's per-coordinate step, which does not care that layer 2 has
   4x more parameters.

3. **Falling loss is not the same as a formed circuit.** If §8 shows `induction_diag_off` near 1.0
   and a small cosine to the construction while CE has dropped substantially, then at this learning
   rate GD has found a **lower-loss solution that is not the induction head**. That is a result, not
   a bug — but such a run cannot be used to argue "GD builds the construction".

4. **Support share vs cosine tells you where it went.** Layer 2 typically keeps its mass at many
   times the uniform baseline inside the construction's support while still having a low cosine to
   it. Compare against the *step-1 gradient*, which sat at 113x. The loss pulls towards the
   circuit's neighbourhood and then settles somewhere else inside it.

5. **Noise density changes how construction-like the answer is.** Lower $\alpha$ (more noise) tends
   to land closer to the construction; at $\alpha$=1.0, with no noise to filter, many solutions do
   the job and GD has no reason to prefer that one.

6. **Compare learning rates before concluding anything.** The lr = 0.005 runs on disk (`ep-300`,
   `ep-3000`) stay in the regime where the pattern can settle. Point `RUN_TAG` at each in turn. If
   the diagonal forms at small lr and not at large, the honest statement is that **the construction
   is what GD finds at small steps, and large steps buy loss at the cost of structure** — worth
   reporting either way.

Figures and GIFs are in `artifacts/figures_full_25k/`, suffixed with the run tag.